Import libraries:

In [20]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

Pull the data from Postgres database:

In [3]:
load_dotenv()

user = os.environ.get('AACT_DB_USER')
password = os.environ.get('AACT_DB_PASSWORD')
host = os.environ.get('AACT_DB_HOST')
port = os.environ.get('AACT_DB_PORT')
dbname = os.environ.get('AACT_DB_NAME')

engine = create_engine(f'postgresql://{user}:{password}@{host}:{port}/{dbname}')

df = pd.read_sql("SELECT COUNT(*) FROM ctgov.studies;", engine)
print(df)

    count
0  598314


Based off of the provided data dictionary (https://aact.ctti-clinicaltrials.org/data_dictionary), select these attributes:
- nct_id: Unique trial identifier
- phase: Trial phase (1, 2, 3, 4)
- enrollment: Target number of participants a trial aims to recruit
- overall_status: Trial completion status
- start_date: Start of trial
- completion_date: End of trial
- agency_class: Trial lead sponsor type
- num_sites: How many locations the trial ran across

In [10]:
query = """
SELECT 
    s.nct_id, 
    s.phase, 
    s.enrollment, 
    s.overall_status, 
    s.start_date, 
    s.completion_date, 
    sp.agency_class AS sponsor_type, 
    COUNT(DISTINCT f.id) AS num_sites
FROM ctgov.studies s
LEFT JOIN ctgov.sponsors sp 
    ON s.nct_id = sp.nct_id AND sp.lead_or_collaborator = 'lead'
LEFT JOIN ctgov.facilities f 
    ON s.nct_id = f.nct_id
WHERE s.overall_status = 'COMPLETED'
  AND s.study_type = 'INTERVENTIONAL'
  AND s.start_date IS NOT NULL
  AND s.completion_date IS NOT NULL
GROUP BY s.nct_id, s.phase, s.enrollment, s.overall_status, 
         s.start_date, s.completion_date, sp.agency_class
"""

df = pd.read_sql(query, engine)

In [11]:
df.head()

,nct_id,phase,enrollment,overall_status,start_date,completion_date,sponsor_type,num_sites
0,NCT00000113,PHASE3,469.0,COMPLETED,1997-09-30,2013-09-30,OTHER,4
1,NCT00000114,PHASE3,NaN,COMPLETED,1984-05-31,1987-06-30,NIH,0
2,NCT00000115,PHASE2,NaN,COMPLETED,1990-12-31,1994-06-30,NIH,0
3,NCT00000116,PHASE3,221.0,COMPLETED,1996-05-31,2002-09-30,NIH,1
4,NCT00000117,PHASE3,NaN,COMPLETED,1995-08-31,1997-12-31,NIH,2


Construct target variable:

In [12]:
df['start_date'] = pd.to_datetime(df['start_date'])
df['completion_date'] = pd.to_datetime(df['completion_date'])
df['duration_days'] = (df['completion_date'] - df['start_date']).dt.days
df = df[df['duration_days'] > 0] # drop any negative/invalid durations

In [13]:
print(df['duration_days'].describe())

count    247587.000000
mean        897.904943
std         886.683518
min           1.000000
25%         287.000000
50%         639.000000
75%        1222.000000
max       38562.000000
Name: duration_days, dtype: float64


In [18]:
df.nlargest(20, 'duration_days')[['nct_id', 'phase', 'start_date', 'completion_date', 'duration_days']]

,nct_id,phase,start_date,completion_date,duration_days
206007,NCT05343208,NA,1916-09-05,2022-04-04,38562
13457,NCT00257140,PHASE2/PHASE3,1931-06-30,1994-07-31,23042
139221,NCT03217539,NA,1977-07-07,2015-12-31,14056
67271,NCT01468883,PHASE3,1979-09-04,2016-11-17,13589
47657,NCT01014039,NA,1983-03-31,2017-01-31,12360
28539,NCT00591643,PHASE1,1977-07-31,2011-03-31,12296
137229,NCT03164291,PHASE3,1984-06-30,2017-05-18,12010
120166,NCT02719678,NA,1982-01-31,2014-09-30,11930
367,NCT00001197,PHASE2,1984-02-07,2015-05-18,11423
9146,NCT00165256,PHASE2,1995-05-15,2026-06-25,11364


Replace 'NA' with proper missing values:

In [19]:
print(df['phase'].value_counts())

phase
NA               126248
PHASE2            31698
PHASE1            29742
PHASE3            24338
PHASE4            20259
PHASE1/PHASE2      7139
PHASE2/PHASE3      3694
EARLY_PHASE1       2614
Name: count, dtype: int64


In [21]:
df['phase'] = df['phase'].replace('NA', np.nan)

Check missing values:

In [25]:
print(df.shape)
print(df.isna().sum())

(245732, 9)
nct_id                  0
phase              126248
enrollment              0
overall_status          0
start_date              0
completion_date         0
sponsor_type            0
num_sites               0
duration_days           0
dtype: int64


Enrollment has low missing values, so we can simply drop these rows:

In [27]:
df = df.dropna(subset=['enrollment'])
print(df.shape)
print(df.isna().sum())

(245732, 9)
nct_id                  0
phase              126248
enrollment              0
overall_status          0
start_date              0
completion_date         0
sponsor_type            0
num_sites               0
duration_days           0
dtype: int64


There is still the issue of the 120,000+ missing values in `phase`. These trials fall outside of the standard Phase 1-4 framework. We will drop them since `phase` will be a predictor for this analysis. This limitation should be acknowledged.

In [28]:
df = df.dropna(subset=['phase'])
print(df.shape)
print(df.isna().sum())

(119484, 9)
nct_id             0
phase              0
enrollment         0
overall_status     0
start_date         0
completion_date    0
sponsor_type       0
num_sites          0
duration_days      0
dtype: int64


We are left with a cleaned dataset of 119,000+ completed interventional clinical trials.

Export the cleaned data:

In [29]:
df.to_csv('clinical_trials_cleaned.csv', index=False)